# **Introduction (Assignment 1, Deadline 23/10/2026 at 08:00 AM)**

Students write text here, explaining:
 - What it means to preprocess data and what is a preprocessing pipeline
 - What are the main artefacts that can be found in the TMS-EEG signal and their causes
 - **Optional**: a personal comment or opinion on the problems posed by preprocessing, based on what you heard in class

**Answer:**

In TMS-EEG, preprocessing describes the process of seperating the signal, which is the fraction of scalp voltage generated by the cerebral cortex in response to the TMS pulse, from the noise, namely the fraction of scalp voltage generated by anything else. 
A preprocessing pipeline is the sequence of transformations applied to the data.
One of the most common artefacts that can be found in TMS-EEG, are muscle artefacts. If TMS is done over the scalp area with large muscles, they most probably gave huge muscle artefacts because TMS activates everything sensitive to magnetism within reach. Another common aretfact could occur when stimulating more frontally, which leads the subject to blink moe frequently than they would under normal conditions. Of course the TMS-pulse itself generates a huge artefact when picked up by the EEG-electrodes over the scalp, so researchers also have to account for this in their preprocessing. 


The problem posed by preprocessing is, that there is no stamdard preprocessing pipeline for EEG or TMS-EEG. What follows is, that every research group all around the world has a different preprocessing pipeline. 
So far, there is also no common ground for the basis on which preprocessing is done. Some groups aim to apply as little transformations to the data as possible in order to avoid changing the data in undesirable or destructive ways. They rather take the risk of having too much noise in their data, tham to loose anything. 
Another opposing approach is to act under the assumption that the actual signal is invisible until all artefacts have been removed. Researchers that believe in this philosophy rather take the risk of deleting some brain activity in order to delete non-brain information, so artefacts, enitrely. 

Generally, two research groups could analyze the same data but get to different results due to completely opposing preprocessing philosophies. This example highlights the importance of documenting all steps in the preprocessing pipeline, when publishing results. 



In [2]:
from pathlib import Path                            # to build a bridge between Python and the filesystem 

import numpy as np                                  # to perform array operations
import matplotlib                                   # this and the following to draw plots
import matplotlib.pyplot as plt
matplotlib.use('Qt5Agg')                            # to make plots interactive


import mne                                          # to read and manipulate EEG data

from scripts import utils                           # custom functions to shorten the code in this notebook

# **1. Basic Preprocessing (Assignment 1, Deadline 23/10/2026 at 08:00 AM)**

Students write text here, listing the operations that we categorise as "basic preprocessing".

If you cannot remember what the steps are, look at the code in the cells below and you will recall. 

**Answer:**

Under basic preprocessing we understand five principle operations. 
**Loading the data**, meaning making it available for the software we want to analyze the data with.

**Inspecting the data** to see its shape and content. Shape describes the number of channels and time-length, resulting in a 2D data format. We also have acquisition parameters, informing about the sampling rate and improtant to understand the content of the data. When inspecting the data, one can also comment on its appearance, informing how visibly dirty the raw data is and to what extent. 

In EEG studies, researchers often coregister data that is not directly related to the actual EEG they are measuring, for example Electromyography, Electrooculogram or Electrocardiogram. When **adjusting the data**, one deletes non-EEG channels that were coregistered.

The other two operations are the **Interpolation of the pulse artefact** and **High-pass Filtering**

## **1.1. Loading data**

**Optional but welcome:** students write here any considerations about this step.

**Answer:**

The data evaluated in this notebook is from a single, healthy subject: **S02C1_M1**.
 The relevant information is distributed across three different files. In this step, first  a path object is defined, which makes the data directory visible for python; then the subject's **vhdr** file is explicitly loaded and saved under the variable "file_of_interest". This file contains metadata, which is all the improtant information on how the data was acquired. 

In [3]:
#defines a path object, make data directory visible for python 
data_dir = Path("data")
subject_and_condition = "S02C1_M1" #name of subject and experimental conditions assigned to this variable, can use it then in following line 

file_of_interest = list(data_dir.glob(pattern=f"{subject_and_condition}.vhdr")) # we are reading the .vhdr file not the actual eeg data
print(file_of_interest, "|", type(file_of_interest), "|", file_of_interest[0])

[PosixPath('data/S02C1_M1.vhdr')] | <class 'list'> | data/S02C1_M1.vhdr


In [4]:
eeg_data = mne.io.read_raw_brainvision(vhdr_fname=file_of_interest[0])

Extracting parameters from data/S02C1_M1.vhdr...
Setting channel info structure...


## **1.2. Adjusting the data**

**Compulsory:** students write here the functional significance of this step (that is, why we do it).

**Answer:**

During the EEG recordings, data from auxiliary channels is often recorded simultaneously, for example electrodes around the eyes to record the electrical activity given my eye blinks of the subject, or electrodes on the wrist to record electrocardiagraphy. 
If one is interested in an EEG-only analysis, any non-EEg channels need to be removed. 
If these channels would not be removed, they might influence the anaylsis and ultimatly lead to false results. 

In [4]:
channels_to_drop = []

for channel_name in eeg_data.ch_names: #contains names of all channels, for every channel we do the operation
    try:
        int(channel_name[-1]) #check wether last letter an integer number or not, if it is, nothing done 
    except ValueError: #if it fails
        if channel_name[-1] != "z": #if it is a letter it can either be the z or the g or i
            channels_to_drop.append(channel_name) # so if its not a z, you take it and append it to a list called channels to drop
print(f"Channels to drop: {channels_to_drop}") #list of channels that we want to drop 

eeg_data = eeg_data.drop_channels(ch_names=channels_to_drop) #drops the ones we dont want and makes a new list with all the eeg channels 
#can print it again then and check wether they were actually removed 

Channels to drop: ['EOG', 'FDI']


Here, electrooculogram (EOG) and electromyography from the first dorsal interosseous (FDI) have been co-recorded. With the code block above, these channels have been dropped from the data set, leaving only actual the EEG-channels to analyze.

## **1.3.Inspecting and Visualizing the data**

**Compulsory:** students write here the functional significance of this step (that is, why we do it).

**Answer:**

In this step, the data is inspected and channel locations are set. The table provides basic information about the acqusition, channels, filters and general variables like the filename, date and object type.

As can be told by the table below,the data contains no information about the shape of the head nor the position of the electrodes on it. There is huge inter-subject variability in head shape and size, a channel can end up on very different places, so we need to put the channels in relation to the head. 
Since the amount of channels is known, MNE's built-in 'easycap-M1' montage can be used and the 3D coordinates of each channel can be projected on a 2-dimensional plane, visualizing their positions.
Projecting the channels onto an average standard headhsape is fundamental for source reconstruction and visulaization. 

In [31]:
easycap_m1_montage = mne.channels.make_standard_montage(kind="easycap-M1") 
easycap_m1_montage.plot()
eeg_data.set_montage(montage=easycap_m1_montage,
                     on_missing="ignore")

<RawBrainVision | S02C1_M1.eeg, 72 x 3307600 (661.5 s), ~87 KiB, data not loaded>

**Compulsory:** here, students describe and comment the plot generated by the code below. In particular, I would like to read:

- What each row represents
- What are the $x$ and $y$ of each row, and their units of measurement
- A description of the artefacts you see, and their likely causes

**Answer:**

The plot generated by the line of code below shows the 70 EEG-channels on its $y$-axis, named after the typical 10-20 system, and time in seconds on the $x$-axis. Therefore, each row represents the elcetrical activity picked up by the corresponding electrode over time. 


3 artefacts are very noticable in the plot:
The first one is repeats itself around every second and is most prominent in the frontal channels. It is much larger in amplitude than the brain activity and lasts only shortly, around 100 ms. These characteristics indicate that we might deal with **eyeblinks**. 
The second one is very prominent across all channels. It has a very high amplitude and frequency, which makes it show up as thick black lines in our plot. The fact that it affects many channels, indicates that it spreads across space. Most likely, this artefact is a **muscle artefact**, since muscles are relatively large. 
Futhermore, there is a a noticable **TMS-pulse artefact**, which appears exactly at the same time as the temporal marker, that marks stumulus onset. 

Additionally, channel 'FC3' continuously shows very high frequency, which might also be an artefact.

In [6]:
eeg_data.plot()

Using qt as 2D backend.


**Compulsory:** here, students define event markers and explain why they are important for EEG data analysis

**Optional:** any comments that go beyond a mere definition are welcome. For example, you could make comments about the structure of event arrays and what it tells us about the nature of events

**Answer:**

Event markers are essential for identifying and segmenting key moments during EEG data collection, for example the TMS-pulse. They are time-stamps that locate events of experimental interest and are usually defined as a set of parameters. 

Event arrays capture these parameters. For TMS-EEG, there is an ordinal number which infroms on the number of pulse that was given. They have a type and a name, which can be either a word or a numeric code, chosen by the researchers. 
There is also the type of occurance stored in units of data points, which means their positions in the EEG matrix; and for how many data points the stimulus lasts. 
Lastly, the number of channels that are affected is stored aswell.


In [7]:
events_from_annotations, events_dict = mne.events_from_annotations(raw=eeg_data) #take that function for use, takes raw data as inpiut and returns 2 things 
events_from_annotations = events_from_annotations[2:]
#can identify TMS pulses on the trace and interpolate 

Used Annotations descriptions: ['New Segment/', 'Stimulus/S 54', 'Stimulus/S255']


## **1.4. Interpolating the pulse artefact**

**Compulsory:** students write here the functional significance of this step (that is, why we do it).

**Answer:**

Once the artifact is identified, it needs to be get rid of. Under the TMS pulse, there would be information generated by the brain, but it is inaccessible due to the TMS pulse hiding it. In this step, the TMS pulse artefact is cut out and the resulting gap is closed via interpolation. 
Interpolation is the act of estimating a function in a given interval, given values from preceeding interval and a following one. Essentially, there is missing data under the TMS pulse artefact and it can be reconstructed via interpolation. 

In [8]:
#divides eeg trace into blocks, centered on the TMS pulses and making a plot 
pre_interpolation_epochs = mne.Epochs(raw=eeg_data,
                                      events=events_from_annotations,
                                      tmin=-1.1,
                                      tmax=0.5,
                                      baseline=None)
pre_interpolation_epochs_tep = pre_interpolation_epochs.average()
pre_interpolation_epochs_tep.plot();

Not setting metadata
200 matching events found
No baseline correction applied
0 projection items activated


In [9]:
# file will contain huge block of EEG data but there are noit the same TMS pulse artefacts, since its already interpolated 
post_interpolation_eeg = mne.io.read_raw(fname="data/post_2_5_interpolation_eeg.fif", 
                                         preload=True)
post_interpolation_epochs = mne.Epochs(raw=post_interpolation_eeg,
                                       events=events_from_annotations,
                                       tmin=-1.1,
                                       tmax=0.5,
                                       baseline=None)
post_interpolation_tep = post_interpolation_epochs.average()
post_interpolation_tep.plot(); #plot to show the results 
#can still see huge fluctations but we got rid of the main artifact 

Opening raw data file data/post_2_5_interpolation_eeg.fif...
    Range : 0 ... 3307599 =      0.000 ...   661.520 secs
Ready.
Reading 0 ... 3307599  =      0.000 ...   661.520 secs...
Not setting metadata
200 matching events found
No baseline correction applied
0 projection items activated


## **1.5. High-pass Filtering**

**Compulsory:** students write here the functional significance of this step 

**Answer:**

EEG signals are a combination of many oscillations at many different frequencies. Whereas some oscillations are genuine brain activity, others represent artefactual sources. For example, sweating of the subject causes very slow oscillations, which should be filtered out. 
A high-pass filter filters the data by leaving out everything that oscillates at a lower frequency than a given threshold, which in this case is 0.1 Hz. Everything that oscillates at a higher frequency than those 0.1 Hz is allowed to pass through. 

In [10]:
post_interpolation_eeg.plot();

In [11]:
#TEP looks like after interpolating the artefact and filtering to keep anything above 0.1 Hz 
#interpolizes high pass filter, that filters data that leaves out everything oscilaating lower than 01. Hz 
post_filtering_eeg = mne.filter.filter_data(data=post_interpolation_eeg.get_data(),
                                            sfreq=post_interpolation_eeg.info["sfreq"],
                                            l_freq=0.1, # our cut-off frequency 
                                            h_freq=None,
                                            method="iir",
                                            iir_params=None,
                                            copy=True,
                                            phase="zero")
post_filtering_eeg = mne.io.RawArray(data=post_filtering_eeg,
                                     info=post_interpolation_eeg.info)
post_filtering_epochs = mne.Epochs(raw=post_filtering_eeg,
                                   events=events_from_annotations,
                                   tmin=-1.1,
                                   tmax=0.5,
                                   baseline=None)
post_filtering_tep = post_filtering_epochs.average()
post_filtering_tep.plot();

Setting up high-pass filter at 0.1 Hz

IIR filter parameters
---------------------
Butterworth highpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 8 (effective, after forward-backward)
- Cutoff at 0.10 Hz: -6.02 dB

Creating RawArray with float64 data, n_channels=70, n_times=3307600
    Range : 0 ... 3307599 =      0.000 ...   661.520 secs
Ready.
Not setting metadata
200 matching events found
No baseline correction applied
0 projection items activated


# **2. Manual Artifact Rejection & Bad Channels Rejection** 
# **(Assignment 2, Deadline 30/10/2026 at 08:00 AM)**

## **2.1. Rationale**

**Compulsory:** students write here the functional significance of this step (that is, why we do it) and expand on the criteria to follow for its execution

**Answer**:

**Manual Artefact Rejection (MAR)** and **Bad Channel Rejection (BDR)** are both essential visual screening steps for cleaning raw EEG data. 

MAR is the act of visually inspecting EEG data and flagging epochs, time windows of EEG data containing an experimental event, for rejection. Its main use cases are removement of large artefacts before or after automatic pre-processing procedures, e.g. Independent Component Analysis (ICA). MAR is usally only applied after ICA for any leftover artefacts, that automated routines might have missed. 
MAR can also be used to remove only the most obvious and unambiguous artefacts when the goal is to act on the data as little as possible, avoiding over-cleaning.
To apply MAR, the epoch must contain an artefact that is circumscribed in time and affects multiple channels. Another criteria is high-frequency and high-amplitude fluctuations exceeding physiologically plausible thresholds of cerebral activity. 

BDR describes the act of visually inspecting the data and flagging whole channels for rejection.
BDR has mainly the same use-cases as MAR. 
The data should again be characterized by high-frequency and high-amplitude fluctuations but these should not be cicumscribed in time; in contrary to reject a channel it has to be affected by the artefact for the majority of the recording period. Furthermore the artefact should affect a single channel, usually resulting from poor scalp-contact.

Once a channel ios excluded, it can be reconstructed via sperical spline interpolation using adjacent electrodes. This approach is viable because EEG volume conduction produces high spatial correlation across neighboring sensors.

## **2.2. Execution**

**Compulsory:** students reject bad epochs and bad channels following appropriate criteria. Provide a list of what you reject and why (for example: _"I choose to reject channel $x$ because it is affected by line noise and epoch $y$ because it is affected by a large mechanical artifact"_)

**Answer**:

**Channels** rejected: C1, C3, FC3, F7, P10, PO10, O1, Oz

**Epochs** rejected: 4, 8, 9, 49 



**FC3** shows very high frequency and high amplitude over the entire recording period. This might be due to bad comtact of the electrode with the scalp. SInce there is yet no technique to recover the channel from this artefact, it is being removed.

**F7** shows a similar pattern, it shows very high frequency over the recording. The ampitude is not as high as  channel FC3, but can be characterized as very unlikely in potraying neural activity as well. 

Moving further in the recording, channels **C1** and **C3** also seems to affected by noise. 
Additionally, **PO10**, **O1** and **Oz** exhibt a high frequency, high amplitide characteristics, indicative for a noise artefact over the recodring period. 

Lastly, **P10** was also decided to be rejected. In contrast to the channels mentioned above, this one shows very low frequency, low amplitude recording. It does not exhibit any fluctuations that its neighboring channels show, which is usually common for EEG recordings, since electrical fields spread over the scalp, meaninng neighboring electrodes should demonstrate similar patterns over the recording. P10 is comparably rather flat, which is an indication that there might have been a general problem with e.g. conductivity in the corresponding electrode. 

Since there is no scientific hypothesis that is beng worked on in this notebook, all these channels can be removed without impacting any experimental design. It is important to notice that the channels, that were chose to be rejected can vary depending on which cortical area is of specific interest.

Moving on to the epochs, **epoch 4** was rejected due to a burst of high-amplitude noise across parietal channels. 
**Epoch 8** and **epoch 9** exhibited high-amplitude, high-frequency contamination, inidcative of the subject moving, causing a gross motor artefact affecting parietal and centro-parietal channels. 
Finally, **epoch 49** has been rejected due to a focal, high-frequency myogenic burst.

It is noticable, that epoch 132 is also heavily affected by a large burst in high-amplitude activity, affecting occipital and parietal channels. Since this artefact stems from channel O1, which has already been flagged for rejection, this epoch is not being removed. Furthermore, epochs 107, 118, 123, 134, 143 and 172 exhibit large disturbances, possibly due to eye-blink artefacts; I still decided not to reject them to avoid excessive preprocessing of the data and the option of applying ICA as the next step.


In [12]:
post_filtering_epochs = mne.read_epochs(fname="data/post_filtering_epo.fif", 
                                        preload=True)

Reading /Users/maylinschneider/Desktop/brainstim-multimodal-main/data/post_filtering_epo.fif ...
    Found the data of interest:
        t =   -1100.00 ...     500.00 ms
        0 CTF compensation matrices available
Not setting metadata
200 matching events found
No baseline correction applied
0 projection items activated


In [13]:

post_filtering_epochs.plot();

In [14]:
post_filtering_epochs

<EpochsFIF | 196 events (all good), -1.1 – 0.5 s (baseline off), ~837.6 MiB, data loaded,
 '54': 196>

In [15]:
print(post_filtering_epochs.info["bads"])

['F7', 'C3', 'C1', 'FC3', 'P10', 'PO10', 'O1', 'Oz']


In [ ]:

dropped_indices = [idx for idx, drop in enumerate(post_filtering_epochs.drop_log) if len(drop) > 0]
print("Dropped epoch indices:", dropped_indices)

Dropped epoch indices: [4, 8, 9, 49]


In [17]:

cleaned_file = "post_filtering_cleaned_epo.fif"
# Save 
post_filtering_epochs.save(cleaned_file, overwrite=True)
print(f"Successfully saved separate cleaned file: {cleaned_file}")

Overwriting existing file.
Overwriting existing file.
Overwriting existing file.
Successfully saved separate cleaned file: post_filtering_cleaned_epo.fif


In [19]:
post_cleaned_epochs = mne.read_epochs(fname="data/post_filtering_cleaned_epo.fif", 
                                        preload=True)

Reading /Users/maylinschneider/Desktop/brainstim-multimodal-main/data/post_filtering_cleaned_epo.fif ...
    Found the data of interest:
        t =   -1100.00 ...     500.00 ms
        0 CTF compensation matrices available
Not setting metadata
196 matching events found
No baseline correction applied
0 projection items activated


In [20]:
post_cleaned_epochs.plot()

## **Interpolating bad channel**

In class it was mentioned, that bad channels can be interpolated, so I wanted to try interpolating one myslef. Since we flagged quite a few channels as bad, including PO10 and Oz, which were neighboring channels, I figured, that it moight be hard to interpolate those; since interpolation of bad channels relies on the signal of the neighboring channels. 
Based on this reasoning I picked channel FC3. FC3 clearly shows heavy noisy activity, which its neighboring channels do not exhibit.
In the following code i reset the....

In [43]:
print(post_filtering_epochs.info["bads"])

[]


In [44]:
original_bads = post_filtering_epochs.info["bads"].copy()

post_filtering_epochs.info["bads"] = ["FC3"]

Here I got an error message indicating that the array cannot contain infinite values or values that are not numbers (NaN).

In [45]:
post_filtering_epochs.interpolate_bads(reset_bads=False)

Setting channel interpolation method to {'eeg': 'spline'}.
Interpolating bad channels.
    Automatic origin fit: head of radius 97.0 mm
Computing interpolation matrix from 69 sensor positions
Interpolating 1 sensors


<EpochsFIF | 196 events (all good), -1.1 – 0.5 s (baseline off), ~837.6 MiB, data loaded,
 '54': 196>

Following from here, I first had to check if the data contains either.

In [46]:

has_nan_data = np.isnan(post_filtering_epochs.get_data()).any()
has_inf_data = np.isinf(post_filtering_epochs.get_data()).any()
print(f"Data contains NaN: {has_nan_data}, Inf: {has_inf_data}")

Data contains NaN: False, Inf: False


Since there were neither infinite nor NaN-values in the data, another option was either missing or NaN 3D coordinates in the montage, since MNE cannot compute the sperical splines which are used for the interpolation process, if an electrode has missing position values. 

In [47]:
# Check channel coordinates 
montage = post_filtering_epochs.get_montage()
if montage is None:
    print("No montage is set!")
else:
    ch_pos = montage.get_positions()["ch_pos"]
    nan_coords = [ch for ch, pos in ch_pos.items() if np.isnan(pos).any()]
    print(f"Channels with NaN coordinates: {nan_coords}")

    # Check FC3 
    print(f"FC3 position: {ch_pos.get('FC3')}")

Channels with NaN coordinates: []
FC3 position: [-0.06185234  0.05713329  0.09376583]


The channel PO3 contains a NaN-value. From the output it seems like the channel was accidently named with a Zero (0) instead of a capital O. 
MNE expects the letter O. With the Zero instead, the operation fails because MNE cannot look up the 3D position, resulting in assigning the channel a NaN-value.
The next plausible step is to rename the channel in accordance to the standard 10-20 montage.

In [48]:
# If it was accidentally named with a zero:
if "P03" in post_filtering_epochs.ch_names:
    post_filtering_epochs.rename_channels({"P03": "PO3"})

# Re-apply the montage so it gets valid coordinates
montage = mne.channels.make_standard_montage("standard_1020")
post_filtering_epochs.set_montage(montage, match_case=False, on_missing="ignore")

<EpochsFIF | 196 events (all good), -1.1 – 0.5 s (baseline off), ~837.6 MiB, data loaded,
 '54': 196>

In [49]:
print(post_filtering_epochs.info["bads"])

['FC3']


Now after properly dhecking for the channels, the function ".interpolate_bads" can be called. I chose to set reset_bads to True, so that the interpolated chennel gets cleared from the bad-channel list automatically.

In [50]:
post_filtering_epochs.interpolate_bads(reset_bads=True)

Setting channel interpolation method to {'eeg': 'spline'}.
Interpolating bad channels.
    Automatic origin fit: head of radius 97.0 mm
Computing interpolation matrix from 69 sensor positions
Interpolating 1 sensors


<EpochsFIF | 196 events (all good), -1.1 – 0.5 s (baseline off), ~837.6 MiB, data loaded,
 '54': 196>

In [51]:
picks = ["FT7", "FC5", "FC3", "FC1", "FCz"]
# (Include only channels that exist in your dataset)
#picks = [ch for ch in picks if ch in post_filtering_epochs.ch_names]

post_filtering_epochs.plot(picks=picks, n_epochs=5, scalings="auto")

After the interpolation the channel is clearly lesss noisy than before and exhibits physiological plausible data similar to its neighboring channels.

<p align="center">
<img src="./files/interpFC3_plot.png" width="1000"/>
</p>

# **3. Independent Component Analysis (ICA)**

# **(Assignment 3, Deadline 06/11/2026 at 08:00 AM)**

## **3.1. Rationale**

**Compulsory:** students write here the functional significance of this step (that is, why we do it) and expand on its rationale, focusing on discussing the independence assumption and its application to brain data.

**Answer:**


## **3.2. Components Selection**

**Compulsory:** select one or more components whose most likely source are eye movements and justify their choices, reporting the number(s) of the selected component(s).
When justifying your choice, discuss both the scalp topography and the time course of the component(s).
 
Selecting no components is a legitimate choice but it must be justified in writing. 

In [3]:
post_rejection_epochs = mne.read_epochs(fname="data/post_rejection_epo.fif", 
                                        preload=True)
ica = mne.preprocessing.read_ica(fname="data/fitted_ica.fif")

Reading /Users/maylinschneider/Desktop/brainstim-multimodal/data/post_rejection_epo.fif ...
    Found the data of interest:
        t =   -1100.00 ...     500.00 ms
        0 CTF compensation matrices available
Not setting metadata
188 matching events found
No baseline correction applied
0 projection items activated
Reading /Users/maylinschneider/Desktop/brainstim-multimodal/data/fitted_ica.fif ...
Now restoring ICA solution ...
Ready.


In [4]:
ica.plot_components(picks=range(30)); 
# takes solution and plots components of 30 independent components, they are ranked from most to least important 
#first component is typical for a eyeblink, lot of red around the noise so intentsity is maximum around eye region and muss less color in rest if the scalp 
#ICA000
#the others could be a lot of different things, its hard to tell whtát it is 
# would report proportion of varaince it explained (the ICA000)

In [5]:
ica.plot_sources(inst=post_rejection_epochs, picks=range(30));
#plots the time series 
#first row, trace of ICA000 shows nothing then a sharp peak very circumscribed in time 
#if you take that info and recombine it with topography you might be sure its an eyeblink 

#ICA13 is a nighmare, maybe due to faulty electrodes 
#the others mainly look like different flavors of line noise, we just have a bad data set 
#there might be some muscle noise where there are certain burtss of activity 
#we are only recquired to do this for eyeblinks because given how dirty data is, its the only clear recognizable. Usually ICA is only done for eyeblinksmaybe for msucle.
#the standard is to do ICA only for eyeblinks and maybe soemtimes muscles

Not setting metadata
188 matching events found
No baseline correction applied
0 projection items activated
Using qt as 2D backend.


In [6]:
ica.exclude = # insert a list of the components to reject. For example, insert [21] if you want to reject component 21 (just a random example)
ica.apply(post_rejection_epochs.load_data())
post_ica_epochs = post_rejection_epochs
post_ica_tep = post_ica_epochs.average() #take average  
post_ica_tep.plot();

SyntaxError: invalid syntax (3195999688.py, line 1)